# DeepTuneIO — Module 03: JobID Extractor

This notebook orchestrates Module 03 using the same dynamic campaign hierarchy
adopted by DeepTuneIO-Indexer and Module 05.

Module 03 runs **off-cluster** and reads only:

`processed/Parser/*.txt`

It generates:

- the JobID/metadata CSV used by later processing;
- a plain JobID manifest that is the only file required by the cluster-side `seff` collector.


## 1. Import Module 03

The notebook does not duplicate the processing logic. The executable implementation
lives in `Module/03_extract_jobids_from_parser.py`.


In [1]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE03_FILE = MODULE_DIR / "03_extract_jobids_from_parser.py"

print("Module directory :", MODULE_DIR)
print("Module 03        :", MODULE03_FILE)

if not MODULE03_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró Module 03 en:\n{MODULE03_FILE}\n\n"
        f"Archivos disponibles en Module:\n"
        + "\n".join(f"  - {p.name}" for p in MODULE_DIR.glob("*"))
    )

spec = spec_from_file_location("deeptuneio_module03", MODULE03_FILE)
module03 = module_from_spec(spec)
sys.modules[spec.name] = module03
spec.loader.exec_module(module03)

print("Version:", module03.VERSION)

Module directory : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\03_Resource_Metadata\Module
Module 03        : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\03_Resource_Metadata\Module\03_extract_jobids_from_parser.py
Version: 3.0.0


In [2]:
from pathlib import Path
import sys
import pandas as pd
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE03_FILE = MODULE_DIR / "03_extract_jobids_from_parser.py"

spec = spec_from_file_location("deeptuneio_module03", MODULE03_FILE)
module03 = module_from_spec(spec)
sys.modules[spec.name] = module03
spec.loader.exec_module(module03)

print("Module directory :", MODULE_DIR)
print("Module 03        :", MODULE03_FILE)
print("Exists           :", MODULE03_FILE.exists())
print("Version          :", module03.VERSION)


Module directory : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\03_Resource_Metadata\Module
Module 03        : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\03_Resource_Metadata\Module\03_extract_jobids_from_parser.py
Exists           : True
Version          : 3.0.0


## 2. Select application and campaign

Only these values change between campaigns.


In [33]:
# ============================================================
# Selección de aplicación, dataset/campaña y escenario
# ============================================================

CAMPAIGNS = {
    "DeepGalaxy": {  # 1
        "DG_bw512_f3c5x64": [ # 1
            "lustre_1ost",   # 1
            "lustre_2ost",   # 2
            "lustre_4ost",   # 3 
            "nfs_SnGPU_e1",  # 4 
        ],
    },

    "DLIOv1": {  # 2
        "HDF5_64bs": [  # 1
            "lustre_ost" # 1
        ],
        "NPZ_64bs": [ # 2
            "lustre_ost"  # 1
        ],
        "TFRecord_256kts_64bs": [ # 3
            "lustre_ost"  # 1
        ],
        "TFRecord_1mts_64bs": [ # 4
            "lustre_ost"  # 1
        ],
    },
}


# ============================================================
# 1. Seleccionar aplicación
# ============================================================

APPLICATION_OPTIONS = list(CAMPAIGNS.keys())

APPLICATION_POS = 2
# 1 = DeepGalaxy
# 2 = DLIOv1

if not 1 <= APPLICATION_POS <= len(APPLICATION_OPTIONS):
    raise ValueError(
        f"APPLICATION_POS debe estar entre 1 y "
        f"{len(APPLICATION_OPTIONS)}"
    )

APPLICATION = APPLICATION_OPTIONS[APPLICATION_POS - 1]


# ============================================================
# 2. Seleccionar dataset/campaña
# ============================================================

DATASET_OPTIONS = list(CAMPAIGNS[APPLICATION].keys())

DATASET_POS = 4

if not 1 <= DATASET_POS <= len(DATASET_OPTIONS):
    raise ValueError(
        f"DATASET_POS debe estar entre 1 y "
        f"{len(DATASET_OPTIONS)}"
    )

DATASET_GROUP = DATASET_OPTIONS[DATASET_POS - 1]


# ============================================================
# 3. Seleccionar escenario
# ============================================================

SCENARIO_OPTIONS = CAMPAIGNS[APPLICATION][DATASET_GROUP]

SCENARIO_POS = 1

if not 1 <= SCENARIO_POS <= len(SCENARIO_OPTIONS):
    raise ValueError(
        f"SCENARIO_POS debe estar entre 1 y "
        f"{len(SCENARIO_OPTIONS)}"
    )

SCENARIO = SCENARIO_OPTIONS[SCENARIO_POS - 1]


# ============================================================
# Mostrar selección
# ============================================================

print("Selección actual")
print("----------------")
print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)


Selección actual
----------------
Application   : DLIOv1
Dataset group : TFRecord_1mts_64bs
Scenario      : lustre_ost


## 3. Build the campaign root

The campaign hierarchy is shared with the other DeepTuneIO modules.


In [34]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"F:\NoteBooks\CORA_DR_UAB"
).resolve()

ARTICLE_ID = "Article_01_Parallel_IO_Analysis"

ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID

CAMPAIGN_ROOT = (
    ARTICLE_ROOT
    / "Data"
    / APPLICATION
    / DATASET_GROUP
    / SCENARIO
)

print("Project root  :", PROJECT_ROOT)
print("Article root  :", ARTICLE_ROOT)
print("Campaign root :", CAMPAIGN_ROOT)
print("Exists        :", CAMPAIGN_ROOT.exists())

Project root  : F:\NoteBooks\CORA_DR_UAB
Article root  : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\lustre_ost
Exists        : True


## 4. Resolve Module 03 paths

The module derives:

- `processed/Parser`
- `results/Resource_Metadata`

from the campaign root.


In [35]:
args = module03.parse_args([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-label", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", "JSC24",
])

paths = module03.resolve_campaign_paths(args)

for key, value in paths.items():
    print(f"{key:14s}: {value}")


campaign_root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost
parser_dir    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\processed\Parser
output_dir    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Metadata


## 5. Execute Module 03


In [36]:
return_code = module03.main([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-label", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", "JSC24",
])

print("Return code:", return_code)


DeepTuneIO — Module 03: JobID Extractor
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost
Application   : DLIOv1
Dataset group : TFRecord_1mts_64bs
Scenario      : lustre_ost
Parser dir    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\processed\Parser
Parser files  : 13
JobIDs        : 13
OK            : 13
Incomplete    : 0
Errors        : 0
CSV           : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Metadata\03_jobids_from_parser_DLIOv1_TFRecord_1mts_64bs_lustre_ost_JSC24.csv
Manifest      : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DLIOv1\TFRecord_1mts_64bs\Lustre_ost\results\Resource_Metadata\03_jobids_DLIOv1_TFRecord_1mts_64bs_lustre_ost_JSC24.txt
Return code: 0


## 6. Inspect generated metadata


In [7]:
RESOURCE_METADATA_DIR = CAMPAIGN_ROOT / "results" / "Resource_Metadata"

csv_files = sorted(
    RESOURCE_METADATA_DIR.glob("03_jobids_from_parser_*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

manifest_files = sorted(
    RESOURCE_METADATA_DIR.glob("03_jobids_*.txt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if csv_files:
    JOBID_CSV = csv_files[0]
    job_df = pd.read_csv(JOBID_CSV)
    print("CSV:", JOBID_CSV)
    display(job_df)
else:
    print("No Module 03 CSV generated yet.")

if manifest_files:
    JOBID_MANIFEST = manifest_files[0]
    print("Manifest:", JOBID_MANIFEST)
    print(JOBID_MANIFEST.read_text(encoding="utf-8"))


CSV: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\lustre_1ost\results\Resource_Metadata\03_jobids_from_parser_DeepGalaxy_DG_bw512_f3c5x64_lustre_1ost_JSC24.csv


,application,scenario,dataset_label,parser_file,parser_path,jobid,input_file,file_format,status,message
0,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,10_DG_N4p16e1_EfficientNetB4_outputbw512hdf5_d...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,8810008,output_bw_512.hdf5,h5,ok,metadata_extracted
1,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,11_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_d...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,5178759,output_bw_512.hdf5,h5,ok,metadata_extracted
2,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,12_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_d...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,5181608,output_bw_512.hdf5,h5,ok,metadata_extracted
3,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,13_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_d...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,5349515,output_bw_512.hdf5,h5,ok,metadata_extracted
4,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,14_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_d...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,8850850,output_bw_512.hdf5,h5,ok,metadata_extracted
5,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,15_DG_N12p48e1_EfficientNetB4_outputbw512hdf5_...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,5171929,output_bw_512.hdf5,h5,ok,metadata_extracted
6,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,16_DG_N12p48e1_EfficientNetB4_outputbw512hdf5_...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,5172393,output_bw_512.hdf5,h5,ok,metadata_extracted
7,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,17_DG_N12p48e1_EfficientNetB4_outputbw512hdf5_...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,8849686,output_bw_512.hdf5,h5,ok,metadata_extracted
8,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,18_DG_N16p64e1_EfficientNetB4_outputbw512hdf5_...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,5333215,output_bw_512.hdf5,h5,ok,metadata_extracted
9,DeepGalaxy,lustre_1ost,DG_bw512_f3c5x64,19_DG_N16p64e1_EfficientNetB4_outputbw512hdf5_...,F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_I...,8787065,output_bw_512.hdf5,h5,ok,metadata_extracted


Manifest: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\lustre_1ost\results\Resource_Metadata\03_jobids_DeepGalaxy_DG_bw512_f3c5x64_lustre_1ost_JSC24.txt
4935078
4969259
4998171
5004132
5005753
5006747
5136097
5145754
5147852
5164453
5171929
5172393
5173520
5178759
5181608
5182586
5183430
5184753
5185397
5186976
5189973
5196039
5317272
5333215
5347797
5348184
5349515
5350177
5352783
5353861
5366451
5369484
8773752
8784568
8787065
8806303
8810008
8849686
8850850



## 7. Quality-control summary

Incomplete Parser metadata is preserved and reported instead of being silently discarded.


In [8]:
if "job_df" in globals() and not job_df.empty:
    display(
        job_df["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="files")
    )

    problems = job_df[job_df["status"] != "ok"]
    if not problems.empty:
        display(problems)


,status,files
0,ok,39


## 8. Next step: Module 04

Only the generated JobID manifest needs to be available on the HPC system.

```text
Module 03 — local
    ↓
03_jobids_....txt
    ↓
Module 04 — HPC login/service node
    ↓
raw/seff/seff_<jobid>.txt
```

The metadata CSV remains local and is later consumed by Module 05.
